# Importing libraries

In [2]:
# Basic libraries
import pandas as pd
from datasets import load_dataset
import time
import pickle


# Classification models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

# Vectorizers
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Utilities and metrics
from itertools import product
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from memory_profiler import memory_usage

# Preprocessing
import nltk
import re

# Download nltk resources
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rafael\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Setting seeds

In [3]:
s1 = 2
s2 = 3
s3 = 5

seeds = [s1, s2, s3]

# Importing datasets

In [4]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

train.info()

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rafael\.cache\huggingface\hub\datasets--cornell-movie-review-data--rotten_tomatoes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 424004.56 examples

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8530 entries, 0 to 8529
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    8530 non-null   object
 1   label   8530 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 133.4+ KB


# Dataset preprocessing

In [5]:
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()

    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)
train['text'] = train['text'].apply(preprocess_text)

# GridSearch implementation

In [6]:
vectorizers = [
    TfidfVectorizer(),
    CountVectorizer()
]

models = {
    'RandomForest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'criterion': ['gini', 'entropy', 'log_loss']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    'MultinomialNB': {
        'model': MultinomialNB(),
        'params': {
            'alpha': [0.01, 0.1, 1.0]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l2']
        }
    },
    'KNeighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'algorithm': ['ball_tree', 'kd_tree', 'brute']
        }
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(),
        'params': {
            'n_estimators': [100, 150, 200],
            'criterion': ['friedman_mse', 'squared_error'],
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 1.0]
        }
    },
    'SGD': {
        'model': SGDClassifier(),
        'params': {
            'alpha': [0.0001, 0.001, 0.01],
            'penalty': ['l2', 'l1', 'elasticnet']
        }
    }
}

In [7]:
columns = ['seed', 'vectorizer', 'model', 'params', 'accuracy', 'training_time', 'prediction_time', 'peak_memory_train', 'peak_memory_prediction']

classes = sorted(train['label'].unique())
for c in classes:
    columns.extend([
        f'precision_class_{c}',
        f'recall_class_{c}',
        f'f1_class_{c}'
    ])

results = pd.DataFrame(columns=columns)

In [ ]:
for seed in seeds:
    print(f"Processing seed: {seed}")
    for vectorizer in vectorizers:
        print(f"Processing vectorizer: {vectorizer.__class__.__name__}")
        for name, info in models.items():
            print(f"Processing model: {name}")

            model = info['model']
            param_grid = info['params']
            param_combinations = product(*param_grid.values())
            
            for combination in param_combinations:
                params = dict(zip(param_grid.keys(), combination))
                model.set_params(**params)
                if 'random_state' in model.get_params():
                    model.set_params(random_state=seed)

                print(f"Training {name} with params {params} and vectorizer {vectorizer.__class__.__name__}")

                pipeline = Pipeline([
                    ('vectorizer', vectorizer),
                    ('model', model)
                ])

                def train_model():
                    pipeline.fit(train['text'], train['label'])

                def predict_model():
                    return pipeline.predict(val['text'])

                # Training Phase
                start_time = time.perf_counter()
                peak_memory_train = memory_usage(train_model, max_usage=True)
                train_time = time.perf_counter() - start_time
                print(f"Training time: {train_time}")
                print(f"Peak memory usage during training: {peak_memory_train} MB")

                # Prediction Phase
                start_time = time.perf_counter()
                peak_memory_pred, y_pred = memory_usage(predict_model, max_usage=True, retval=True)
                prediction_time = time.perf_counter() - start_time
                print(f"Prediction time: {prediction_time}")
                print(f"Peak memory usage during prediction: {peak_memory_pred} MB")
                
                accuracy = accuracy_score(val['label'], y_pred)
                precisions, recalls, f1s, supports = precision_recall_fscore_support(val['label'], y_pred, average=None, labels=classes, zero_division=0)

                result_dict = {
                    'seed': seed,
                    'vectorizer': vectorizer.__class__.__name__,
                    'model': name,
                    'params': params,
                    'accuracy': accuracy,
                    'training_time': train_time,
                    'prediction_time': prediction_time,
                    'peak_memory_train': peak_memory_train,
                    'peak_memory_prediction': peak_memory_pred,
                }

                for i, c in enumerate(classes):
                    result_dict[f'precision_class_{c}'] = precisions[i]
                    result_dict[f'recall_class_{c}'] = recalls[i]
                    result_dict[f'f1_class_{c}'] = f1s[i]

                result = pd.DataFrame([result_dict])
                results = pd.concat([results, result], ignore_index=True)
                print("-"*100)

results.to_csv('results/results_sklearn_binary2.csv', index=False)

# Process results

In [9]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   seed                    396 non-null    object 
 1   vectorizer              396 non-null    object 
 2   model                   396 non-null    object 
 3   params                  396 non-null    object 
 4   accuracy                396 non-null    float64
 5   training_time           396 non-null    float64
 6   prediction_time         396 non-null    float64
 7   peak_memory_train       396 non-null    float64
 8   peak_memory_prediction  396 non-null    float64
 9   precision_class_0       396 non-null    float64
 10  recall_class_0          396 non-null    float64
 11  f1_class_0              396 non-null    float64
 12  precision_class_1       396 non-null    float64
 13  recall_class_1          396 non-null    float64
 14  f1_class_1              396 non-null    fl

In [10]:
results.head()

,seed,vectorizer,model,params,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,recall_class_0,f1_class_0,precision_class_1,recall_class_1,f1_class_1
0,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'gini'}",0.686679,4.771122,1.054512,394.238281,392.125000,0.660225,0.769231,0.710572,0.723596,0.604128,0.658487
1,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'entropy'}",0.694184,4.523263,1.024106,392.199219,389.671875,0.664025,0.786116,0.719931,0.737931,0.602251,0.663223
2,2,TfidfVectorizer,RandomForest,"{'n_estimators': 50, 'criterion': 'log_loss'}",0.694184,4.544824,1.046450,391.890625,389.589844,0.664025,0.786116,0.719931,0.737931,0.602251,0.663223
3,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'gini'}",0.694184,8.957143,1.082771,422.070312,420.007812,0.666132,0.778612,0.717993,0.733634,0.609756,0.665984
4,2,TfidfVectorizer,RandomForest,"{'n_estimators': 100, 'criterion': 'entropy'}",0.700750,8.625828,1.106410,420.015625,417.371094,0.668239,0.797373,0.727117,0.748837,0.604128,0.668744


In [11]:
results['params'] = results['params'].astype(str)
results_avg_seed = results.groupby(['model', 'vectorizer', 'params']).mean().reset_index()
results_avg_seed['f1_avg'] = results_avg_seed[[col for col in results_avg_seed.columns if 'f1_class' in col]].mean(axis=1)
results_avg_seed

,model,vectorizer,params,seed,accuracy,training_time,prediction_time,peak_memory_train,peak_memory_prediction,precision_class_0,recall_class_0,f1_class_0,precision_class_1,recall_class_1,f1_class_1,f1_avg
0,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.01}",3.333333,0.544090,1.634768,1.288052,437.552083,432.941406,0.653595,0.187617,0.291545,0.525739,0.900563,0.663900,0.477723
1,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 0.1}",3.333333,0.590056,1.506108,1.261543,437.575521,433.701823,0.692000,0.324578,0.441890,0.558824,0.855535,0.676056,0.558973
2,AdaBoost,CountVectorizer,"{'n_estimators': 100, 'learning_rate': 1.0}",3.333333,0.633208,1.567179,1.367073,437.664062,433.996094,0.594667,0.836773,0.695246,0.724684,0.429644,0.539458,0.617352
3,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.01}",3.333333,0.550657,2.035664,1.058116,437.720052,435.153646,0.668750,0.200750,0.308802,0.529801,0.900563,0.667130,0.487966
4,AdaBoost,CountVectorizer,"{'n_estimators': 150, 'learning_rate': 0.1}",3.333333,0.605066,1.907025,1.207950,437.496094,434.920573,0.697183,0.371482,0.484700,0.571611,0.838649,0.679848,0.582274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'rbf'}",3.333333,0.764540,6.418873,1.038377,691.209635,455.520833,0.753597,0.786116,0.769513,0.776471,0.742964,0.759348,0.764431
128,SVC,TfidfVectorizer,"{'C': 1, 'kernel': 'sigmoid'}",3.333333,0.747655,4.471085,0.948226,675.070312,455.342448,0.740000,0.763602,0.751616,0.755814,0.731707,0.743565,0.747591
129,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'linear'}",3.333333,0.724203,7.001554,0.790195,662.924479,455.852865,0.723364,0.726079,0.724719,0.725047,0.722326,0.723684,0.724202
130,SVC,TfidfVectorizer,"{'C': 10, 'kernel': 'rbf'}",3.333333,0.764540,10.997418,1.061838,661.363281,457.033854,0.757299,0.778612,0.767808,0.772201,0.750469,0.761180,0.764494


In [12]:
best_result = results_avg_seed.loc[results_avg_seed['f1_avg'].idxmax()]

best_model = models[best_result['model']]['model']
best_params = eval(best_result['params'])
best_model.set_params(**best_params)

best_result_vectorizer = eval(best_result['vectorizer'])()

pipeline = Pipeline([
    ('vectorizer', best_result_vectorizer),
    ('model', best_model)
])

pipeline.fit(train['text'], train['label'])
y_pred = pipeline.predict(test['text'])

accuracy = accuracy_score(test['label'], y_pred)
precisions, recalls, f1s, supports = precision_recall_fscore_support(test['label'], y_pred, average=None, labels=classes, zero_division=0)

print(f"Best model: {best_result['model']}")
print(f"Best model params: {best_result['params']}")
print(f"Best vectorizer: {best_result['vectorizer']}")
print(f"Best accuracy: {accuracy}\n")

for i, c in enumerate(classes):
    print(f"Class {c}")
    print(f"Precision: {precisions[i]}")
    print(f"Recall: {recalls[i]}")
    print(f"F1: {f1s[i]}")
    print(f"Support: {supports[i]}\n")

Best model: LogisticRegression
Best model params: {'C': 1, 'penalty': 'l2'}
Best vectorizer: TfidfVectorizer
Best accuracy: 0.7467166979362101

Class 0
Precision: 0.7485822306238186
Recall: 0.7429643527204502
F1: 0.7457627118644068
Support: 533

Class 1
Precision: 0.74487895716946
Recall: 0.7504690431519699
F1: 0.7476635514018691
Support: 533



In [13]:
with open('models/best_model_sklearn_binary2.pkl', 'wb') as f:
    pickle.dump(pipeline, f)